# 01 — Phase 1: GRPO on GSM8K (stage A) + dashboard logging

Tommy's spec: GRPO on Qwen2.5-0.5B, 512 GSM8K questions, exact-answer reward,
checkpoints at **0 / 25 / 50 / 100 / 200 updates**.

**GPU:** A100 recommended (generation-heavy). Record compute units before/after
in `compute_log.md` + screenshot the Colab GPU panel (Tommy's hard requirement).

**Deviations / choices to mirror into the Research Doc:**
- base model (not Instruct) — briefing default, open question #1
- β(KL)=0 — danger-zone flavor (briefing §6), open question #4
- lr 1e-6, temperature 0.7, 8 generations/prompt, 8 prompts/update — suggestions
  from the spec kept as-is; any edit to CONFIG below must be justified here.
- **2026-07-09 precision fix:** weights load in **float32** (master) with
  `bf16=True` autocast, plus an update-effectiveness sentinel every 25 steps.
  Pure-bf16 weights at lr 1e-6 round every AdamW update to zero — verified on
  the win4070 cuda run `local_cuda_grpo_gsm8k_6a075c15808e` (a 200-update
  no-op; see `eaaj-pilot-win4070/WIN4070_RUN_ANALYSIS.md`). CONFIG gained
  `master_dtype` so the Colab run-dir hash reflects the fixed recipe.

**Pre-compute gate:** the notebook samples 8 fixed prompt groups before training and aborts if exact reward has no within-group variance. A failure is evidence for Slack open question #1, not permission to silently change the model or reward.

In [ ]:
%pip install -q trl==1.6.0 transformers==5.13.0 datasets==5.0.0 accelerate==1.14.0 pytest==8.4.2 numpy==2.3.5 scipy==1.16.3 pandas==2.3.3 matplotlib==3.10.6

In [ ]:
import json, os, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/eaaj-pilot")
else:
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
from src.repro import config_hash, runtime_manifest, set_active_run

PILOT = json.loads(Path("pilot_config.json").read_text())
CONFIG = {"model": PILOT["model_id"],
          "model_revision": PILOT["model_revision"],
          "seed": PILOT["seed"], **PILOT["stage_a"], "bf16": True,
          "master_dtype": "float32"}
CONFIG_HASH = config_hash(CONFIG)
RUN_DIR = PROJECT_DIR / "outputs" / f"grpo_gsm8k_{CONFIG_HASH}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
if (RUN_DIR / "dashboard.jsonl").exists():
    raise RuntimeError(f"{RUN_DIR} already contains training logs; never append a second run")
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=1))
env = runtime_manifest(PROJECT_DIR, CONFIG)
(RUN_DIR / "manifest.json").write_text(json.dumps(env, indent=1))
set_active_run(PROJECT_DIR, RUN_DIR)
print("run dir:", RUN_DIR, "| GPU:", env["gpu"])
print(">> compute_log.md: record units BEFORE starting")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import GRPOConfig, GRPOTrainer

from src.callbacks import (ExactAnswerEvalCallback, JsonlDashboardLogger,
                           SaveAtSteps, UpdateEffectivenessSentinel)
from src.data import gsm8k_eval_set, gsm8k_grpo_dataset
from src.preflight import sparse_reward_preflight
from src.reward import exact_answer_reward

assert torch.cuda.is_available(), "Phase 1 must run on a Colab GPU"
assert torch.cuda.is_bf16_supported(), "Default recipe requires L4/A100 bf16; log any T4/fp16 deviation"
set_seed(CONFIG["seed"])
tok = AutoTokenizer.from_pretrained(CONFIG["model"], revision=CONFIG["model_revision"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# 2026-07-09 fix: load fp32 MASTER weights; bf16=True autocasts the compute.
# Loading the params themselves in bf16 makes every lr=1e-6 AdamW update
# round to zero (bf16 ulp ~|w|*2^-8 >> 1e-6). Verified on the win4070 run
# local_cuda_grpo_gsm8k_6a075c15808e (200 no-op updates) - see
# eaaj-pilot-win4070/WIN4070_RUN_ANALYSIS.md.
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model"], revision=CONFIG["model_revision"],
    dtype=torch.float32).to("cuda")

# checkpoint 0 = untouched base model
ckpt0 = RUN_DIR / "ckpt-0"
if not ckpt0.exists():
    model.save_pretrained(ckpt0, safe_serialization=True)
    tok.save_pretrained(ckpt0)
print("saved", ckpt0)

In [ ]:
train_ds = gsm8k_grpo_dataset()
eval_prompts, eval_golds, eval_metadata = gsm8k_eval_set(return_metadata=True)
print(f"train questions: {len(train_ds)} | eval slice: {len(eval_prompts)}")

# No-update budget gate: sparse exact reward must vary within at least one
# sampled GRPO group, otherwise all advantages are zero.
preflight = sparse_reward_preflight(
    model, tok, train_ds["prompt"][:8], train_ds["answer"][:8],
    num_generations=CONFIG["num_generations"],
    temperature=CONFIG["temperature"], top_p=CONFIG["top_p"],
    max_new_tokens=CONFIG["max_completion_length"])
(RUN_DIR / "sparse_reward_preflight.json").write_text(json.dumps(preflight, indent=1))
print({k: preflight[k] for k in ("n_correct", "groups_with_reward_variance", "has_grpo_signal")})
if not preflight["has_grpo_signal"]:
    raise RuntimeError("No GRPO learning signal. Flag base-vs-Instruct open question #1 in Slack; do not spend 200 updates.")
set_seed(CONFIG["seed"])  # preflight sampling must not shift formal training RNG

args = GRPOConfig(
    output_dir=str(RUN_DIR / "trainer"), seed=CONFIG["seed"],
    max_steps=CONFIG["max_steps"], learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_generations=CONFIG["num_generations"], beta=CONFIG["beta"],
    temperature=CONFIG["temperature"], top_p=CONFIG["top_p"],
    max_completion_length=CONFIG["max_completion_length"],
    bf16=CONFIG["bf16"], logging_steps=1, save_strategy="no", report_to="none")

trainer = GRPOTrainer(
    model=model, args=args, train_dataset=train_ds,
    reward_funcs=exact_answer_reward, processing_class=tok,
    callbacks=[
        JsonlDashboardLogger(RUN_DIR / "dashboard.jsonl"),
        UpdateEffectivenessSentinel(RUN_DIR / "update_sentinel.jsonl", every=25),
        SaveAtSteps(CONFIG["checkpoint_steps"][1:], RUN_DIR, tokenizer=tok),
        ExactAnswerEvalCallback(eval_prompts, eval_golds,
                                RUN_DIR / "gsm8k_eval.jsonl",
                                every=CONFIG["eval_every"], also_at_step0=True,
                                item_metadata=eval_metadata),
    ])

In [ ]:
t0 = time.time()
trainer.train()
wall = time.time() - t0
(RUN_DIR / "wall_clock.json").write_text(json.dumps(
    {"phase": "grpo_gsm8k_200_updates", "wall_seconds": wall, "gpu": env["gpu"]}, indent=1))
print(f"training done in {wall/3600:.2f} h")
print(">> compute_log.md: record units AFTER + screenshot the GPU panel NOW")

In [ ]:
# quick sanity: reward curve summary from the dashboard log
import pandas as pd
rows = [json.loads(l) for l in open(RUN_DIR / "dashboard.jsonl")]
df = pd.DataFrame(rows)
reward_col = next(c for c in df.columns if c.endswith("reward") or c == "reward")
print(df[["step", reward_col]].groupby(df.step // 25 * 25).mean())